In [1]:
import pandas as pd
import importlib
import sys
import os

sys.path.append(
    os.path.abspath("../data_prepration")
)
import load_data
importlib.reload(load_data)

<module 'load_data' from 'e:\\DataAnalysis\\_Projects\\proj_17\\product_feature_performance_analysis\\data_prepration\\load_data.py'>

### **subscriptions table duplicates check**

In [2]:
subscriptions = load_data.load_subscriptions()
dups_count = subscriptions.duplicated().sum()
dups_count

np.int64(0)

### **subscriptions table missing value check**

In [3]:
missing_valuse_count = subscriptions.isna().sum()
missing_valuse_count

subscription_id            0
user_id                    0
subscription_type          0
subscription_start_date    0
subscription_end_date      0
is_active                  0
dtype: int64

### **subscriptions table primary uniqueness check**

In [4]:
subscriptions["subscription_id"].is_unique

True

### **subscriptions orphan records by user_id**

In [5]:
users = load_data.load_users()
orphan_subscriptions_by_user = (
    ~subscriptions["user_id"].isin(users["user_id"])
).sum()
orphan_subscriptions_by_user

np.int64(0)

### **subscriptions subscription_start_date and subscription_end_date validity**

In [20]:
subscriptions["subscription_end_date"].gt(subscriptions["subscription_start_date"]).all()

np.True_

### **subscriptions type distribution**

In [22]:
subscriptions_type_distribution = (
    subscriptions
        .groupby("subscription_type")
        .agg(
            subscriptions_count = ("subscription_id", "count")
        )
)
subscriptions_type_distribution["percentage"] = (subscriptions_type_distribution["subscriptions_count"] / subscriptions_type_distribution["subscriptions_count"].sum()).round(4)
subscriptions_type_distribution.to_csv("../../output/subscriptions_type_distribution.csv")
subscriptions_type_distribution

,subscriptions_count,percentage
subscription_type,,
Monthly,64155,0.6486
Quarterly,24747,0.2502
Yearly,10012,0.1012


### **subscriptions active status distribution**

In [24]:
subscriptions_active_status_distribution = (
    subscriptions
        .groupby("is_active")
        .agg(
            subscriptions_count = ("subscription_id", "count")
        )
)
subscriptions_active_status_distribution["percentage"] = (subscriptions_active_status_distribution["subscriptions_count"] / subscriptions_active_status_distribution["subscriptions_count"].sum()).round(4)
subscriptions_active_status_distribution.to_csv("../../output/subscriptions_active_status_distribution.csv")
subscriptions_active_status_distribution

,subscriptions_count,percentage
is_active,,
False,67442,0.6818
True,31472,0.3182


### **subscription start distribution over time**

In [27]:
subscriptions_distribution_overtime = (
    subscriptions
        .assign(
            Month_Number = lambda df: df["subscription_start_date"].dt.month,
            Month_Name = lambda df: df["subscription_start_date"].dt.month_name(),
            Month_Name_Short = lambda df: df["subscription_start_date"].dt.strftime("%b"),
            Date_Label = lambda df: df["subscription_start_date"].dt.strftime("%b-%y")
        )
        .groupby(["Month_Number", "Month_Name", "Month_Name_Short", "Date_Label"])
        .agg(
            subscriptions_count = ("subscription_id", "count")
        )
)
subscriptions_distribution_overtime["percentage"] = (subscriptions_distribution_overtime["subscriptions_count"] / subscriptions_distribution_overtime["subscriptions_count"].sum()).round(4)
subscriptions_distribution_overtime.to_csv("../../output/subscriptions_distribution_overtime.csv")
subscriptions_distribution_overtime

,,,,subscriptions_count,percentage
Month_Number,Month_Name,Month_Name_Short,Date_Label,,
1,January,Jan,Jan-25,1415,0.0143
2,February,Feb,Feb-25,3685,0.0373
3,March,Mar,Mar-25,7025,0.0710
4,April,Apr,Apr-25,8129,0.0822
5,May,May,May-25,8485,0.0858
6,June,Jun,Jun-25,8164,0.0825
7,July,Jul,Jul-25,8252,0.0834
8,August,Aug,Aug-25,8423,0.0852
9,September,Sep,Sep-25,8102,0.0819
